# Lab 8 Simulation

This notebook demonstrates the simplified CPI loading workflow and compares the three loading methods: `append`, `trunc`, and `inc`.

## Setup

The notebook runs from `lab8/notebooks/`, so we first add the repo root to `sys.path` and import the helper functions from `lab8/cpi_lab.py`.

In [1]:
from pathlib import Path
import os
import sys

import duckdb
import pandas as pd

repo_root = Path.cwd().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)

from lab8.cpi_lab import benchmark_daily_range, ensure_snapshots, initialize_from_snapshot, summarize_tables, update_from_snapshot

## Generate snapshots and initialize the database

In [2]:
ensure_snapshots()
initialize_from_snapshot()
summarize_tables()

,table_name,row_count,min_date,max_date
0,cpi_append,924,1947-01-01,2023-12-01
1,cpi_trunc,924,1947-01-01,2023-12-01
2,cpi_inc,924,1947-01-01,2023-12-01


At this point, all three tables should contain the same rows because they were initialized from `PCPI24M1.csv`.

## Apply the February 2025 update with each loading method

In [3]:
for method in ("append", "trunc", "inc"):
    update_from_snapshot(method)

summarize_tables()

,table_name,row_count,min_date,max_date
0,cpi_append,937,1947-01-01,2025-01-01
1,cpi_trunc,937,1947-01-01,2025-01-01
2,cpi_inc,937,1947-01-01,2025-01-01


The row counts are now the same, but the values do not necessarily match. `append` only inserts new dates, while `trunc` and `inc` incorporate revisions.

In [4]:
con = duckdb.connect("lab8/lab8.duckdb")

comparison = con.execute("""
SELECT
    COUNT(*) FILTER (WHERE ROUND(a.cpi, 6) <> ROUND(i.cpi, 6)) AS append_vs_inc_mismatches,
    COUNT(*) FILTER (WHERE ROUND(t.cpi, 6) <> ROUND(i.cpi, 6)) AS trunc_vs_inc_mismatches
FROM cpi_append a
JOIN cpi_trunc t USING (dates)
JOIN cpi_inc i USING (dates)
""").fetchdf()

sample_differences = con.execute("""
SELECT
    a.dates,
    a.cpi AS append_cpi,
    t.cpi AS trunc_cpi,
    i.cpi AS inc_cpi
FROM cpi_append a
JOIN cpi_trunc t USING (dates)
JOIN cpi_inc i USING (dates)
WHERE ROUND(a.cpi, 6) <> ROUND(i.cpi, 6)
ORDER BY a.dates DESC
LIMIT 10
""").fetchdf()

con.close()

comparison

,append_vs_inc_mismatches,trunc_vs_inc_mismatches
0,60,0


In [5]:
sample_differences

,dates,append_cpi,trunc_cpi,inc_cpi
0,2023-12-01,308.850,308.735,308.735
1,2023-11-01,307.917,308.087,308.087
2,2023-10-01,307.619,307.653,307.653
3,2023-09-01,307.481,307.374,307.374
4,2023-08-01,306.269,306.138,306.138
5,2023-07-01,304.348,304.615,304.615
6,2023-06-01,303.841,304.099,304.099
7,2023-05-01,303.294,303.316,303.316
8,2023-04-01,302.918,302.858,302.858
9,2023-03-01,301.808,301.643,301.643


## Benchmark a daily simulation

This simulates running the loading script every day from January 1, 2024 through February 28, 2025.

In [6]:
benchmark_results = pd.DataFrame(benchmark_daily_range(start="2024-01-01", end="2025-02-28"))
benchmark_results

,method,seconds,row_count,revised_rows_vs_truth
0,append,2.132584,937,72
1,trunc,3.597728,937,0
2,inc,3.383904,937,0


## Discussion

- `append` is the simplest method, but it misses historical revisions.
- `trunc` is fully consistent because it reloads the full snapshot every time.
- `inc` is also consistent here, while avoiding a full table rewrite.
- For this revised time-series dataset, `inc` is usually the best tradeoff between correctness and efficiency.